In [3]:
import os
import shutil
import time
from datetime import datetime, timedelta

# مسیر مبدا و مقصد
source_dir = r"D:\PT\edge_rl_3"
dest_dir = r"C:\Users\Admin\Desktop\edge_rl_3"

# محاسبه زمان ۳ ساعت قبل
cutoff_time = datetime.now() - timedelta(hours=3)

# تبدیل به timestamp برای مقایسه با mtime
cutoff_timestamp = cutoff_time.timestamp()

# پیمایش دایرکتوری مبدا
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            
            # دریافت زمان آخرین ویرایش
            mtime = os.path.getmtime(file_path)
            
            # اگر فایل در ۳ ساعت اخیر ویرایش شده
            if mtime >= cutoff_timestamp:
                # محاسبه مسیر نسبی نسبت به مبدا
                rel_path = os.path.relpath(root, source_dir)
                dest_subdir = os.path.join(dest_dir, rel_path)
                
                # ساخت پوشه مقصد در صورت نیاز
                os.makedirs(dest_subdir, exist_ok=True)
                
                # مسیر کامل فایل مقصد
                dest_file = os.path.join(dest_subdir, file)
                
                # کپی فایل
                shutil.copy2(file_path, dest_file)
                print(f"کپی شد: {file_path} -> {dest_file}")

کپی شد: D:\PT\edge_rl_3\algorithms\base.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\base.py
کپی شد: D:\PT\edge_rl_3\algorithms\greedy\greedy_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\greedy\greedy_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\hpa\hpa_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\hpa\hpa_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\ppo\env.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\ppo\env.py
کپی شد: D:\PT\edge_rl_3\algorithms\ppo\ppo_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\ppo\ppo_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\voila\voila_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\voila\voila_algorithm.py
کپی شد: D:\PT\edge_rl_3\common\metrics.py -> C:\Users\Admin\Desktop\edge_rl_3\common\metrics.py
کپی شد: D:\PT\edge_rl_3\common\models.py -> C:\Users\Admin\Desktop\edge_rl_3\common\models.py
کپی شد: D:\PT\edge_rl_3\k8s_adapter\realtime_dispatcher.py -> C:\Users\Admin\Desktop\edge_rl

In [ ]:
# check_placement.py
from common.config import CFG
from common.models import Server
from data.loader import load_train
from algorithms.ppo.optimal_placement import aggregate_training_demand, solve_optimal_server_selection

servers = {}
for sid, info in CFG.server_info.items():
    prof = CFG.server_profiles[info["profile"]]
    servers[sid] = Server(id=sid, profile=info["profile"], lat=info["lat"], long=info["long"],
                           capacity=info["capacity"], p_idle=prof["p_idle"], p_max=prof["p_max"])

train_events = load_train()
demand_points = aggregate_training_demand(train_events)

# می‌تونی وزن‌ها رو عوض کنی تا ببینی نتیجه چطور تغییر می‌کنه
selected = solve_optimal_server_selection(
    servers, demand_points,
    w_count=0.2, w_energy=1.0, w_distance=0.2,
)

total_cpu_needed = sum(s["cpu_demand"] for s in CFG.services_info.values())
total_selected_capacity = sum(servers[sid].capacity for sid in selected)
total_idle_power = sum(servers[sid].p_idle for sid in selected)

print(f"تعداد نقاط تقاضا: {len(demand_points)}")
print(f"تعداد سرور انتخاب‌شده: {len(selected)}")
print(f"مجموع ظرفیت: {total_selected_capacity} (نیاز: {total_cpu_needed})")
print(f"مجموع توان idle: {total_idle_power}W")
print(f"سرورهای انتخاب‌شده: {sorted(selected)}")
for sid in sorted(selected):
    info = CFG.server_info[sid]
    print(f"  سرور {sid}: profile={info['profile']}, capacity={info['capacity']}, "
          f"p_idle={CFG.server_profiles[info['profile']]['p_idle']}W")

تعداد نقاط تقاضا: 1487
تعداد سرور انتخاب‌شده: 2
مجموع ظرفیت: 260 (نیاز: 248)
مجموع توان idle: 150W
سرورهای انتخاب‌شده: [2, 9]
  سرور 2: profile=edge_small, capacity=60, p_idle=40W
  سرور 9: profile=large, capacity=200, p_idle=110W


In [2]:
# verify_all_fixes.py
"""بررسی وجود همه‌ی فیکس‌های این گفتگو، قبل از شروع آموزش نهایی PPO."""
import inspect

checks = []

def check(name, condition):
    status = "✓" if condition else "✗ MISSING"
    checks.append((name, condition))
    print(f"[{status}] {name}")

# --- دور ۱: مدل دامنه و متریک ---
from common.models import Server
src = inspect.getsource(Server.instantaneous_utilization)
check("۱.۱ DRAINING در utilization", "DRAINING" in src)

from common.metrics import MetricsCollector
src = inspect.getsource(MetricsCollector.record_snapshot)
check("۱.۲ load_balance_cv با ۱ سرور فعال", "len(active) == 1" in src)

# --- دور ۲: capacity-starved ---
from algorithms.base import AlgorithmBase
src = inspect.getsource(AlgorithmBase._capacity_starved_services)
check("۲.۱ capacity_starved شامل BOOTING", "BOOTING" in src)
check("۲.۱ threshold پارامتری شده", "occ_threshold" in src)

from algorithms.voila.voila_algorithm import VoilaAlgorithm
src = inspect.getsource(VoilaAlgorithm.provision_decision)
check("۲.۲ Voila با OCC_UP_THRESHOLD خودش", "OCC_UP_THRESHOLD" in src and "_capacity_starved_services" in src)

from simulator.engine import SimulationEngine
src = inspect.getsource(SimulationEngine._any_service_capacity_starved)
check("۲.۳ engine._any_service_capacity_starved شامل BOOTING", "BOOTING" in src)

# --- دور ۳: n_ready_replicas در snapshot ---
src = inspect.getsource(SimulationEngine._build_metrics_snapshot)
check("۳ n_ready_replicas در snapshot معمولی", '"n_ready_replicas"' in src)
src_ro = inspect.getsource(SimulationEngine._build_metrics_snapshot_readonly)
check("۳ n_ready_replicas در snapshot readonly", '"n_ready_replicas"' in src_ro)

# --- دور ۴: action mask ---
from algorithms.ppo.env import EdgeResourceEnv
src = inspect.getsource(EdgeResourceEnv._any_server_can_host)
check("۴.۱ mask can_up چک ACTIVE", "ServerState.ACTIVE" in src)
src = inspect.getsource(EdgeResourceEnv.action_masks)
check("۴.۱ mask can_down از n_ready/mature_ready", "ready_replicas" in src.lower() or "n_ready_replicas" in src or "n_mature_ready_replicas" in src)

from algorithms.ppo.ppo_algorithm import PPOAlgorithm
src = inspect.getsource(PPOAlgorithm._build_action_masks)
check("۴.۲ ppo_algorithm mask چک ACTIVE", "ServerState.ACTIVE" in src)

# --- دور ۵: demand_centroid در لحظه‌ی ورود ---
src = inspect.getsource(SimulationEngine._handle_arrival)
n_appends = src.count("_recent_positions[req.service_id].append")
check("۵ demand_centroid فقط یک‌بار append میشه (نه دوبار)", n_appends == 1)

# --- دور ۶: ENERGY_RESYNC + drain دینامیک ---
from simulator.events import EventType
check("۶.۱ ENERGY_RESYNC در EventType", hasattr(EventType, "ENERGY_RESYNC"))
check("۶.۲ ENERGY_RESYNC در step()", "ENERGY_RESYNC" in inspect.getsource(SimulationEngine.step))
src = inspect.getsource(SimulationEngine._start_replica_drain)
check("۶.۳ drain دینامیک (available_at)", "available_at" in src)

# --- دور بعد: خودارجاعی TURN_ON/TURN_OFF ---
check("۷.۱ _was_turn_on_necessary_audit موجود", hasattr(SimulationEngine, "_was_turn_on_necessary_audit"))
check("۷.۲ _was_turn_off_necessary_audit موجود", hasattr(SimulationEngine, "_was_turn_off_necessary_audit"))

# --- proximity در فاز ۳ ---
try:
    from k8s_adapter.realtime_dispatcher import RealtimeEngine
    src = inspect.getsource(RealtimeEngine.__init__)
    check("۸ _tick_proximity_violated در realtime_dispatcher", "_tick_proximity_violated" in src)
except ImportError:
    print("[SKIP] k8s_adapter نیاز به کتابخانه‌ی kubernetes/redis دارد - چک نشد")

# --- bias سرور اول ---
src = inspect.getsource(EdgeResourceEnv.step)
check("۹.۱ رفع bias در env.py (نه break ساده)", "np_random.choice" in src or "random" in src.lower())
src = inspect.getsource(PPOAlgorithm._predict_and_cache)
check("۹.۲ رفع bias در ppo_algorithm.py", "_tie_break_rng" in src or "random" in src.lower())

# --- محافظت سن replica ---
src = inspect.getsource(SimulationEngine._apply_scale_decision)
check("۱۰ محافظت سن replica (created_at/mature)", "created_at" in src or "mature" in src)

# --- کالیبراسیون ---
from common import state_builder
check("۱۱.۱ _NORM_RESPONSE_TIME_SEC بازکالیبره", abs(state_builder._NORM_RESPONSE_TIME_SEC - 300.0) > 1)
check("۱۱.۲ _NORM_ENERGY_JOULE بازکالیبره", abs(state_builder._NORM_ENERGY_JOULE - 12000.0) > 1)
check("۱۱.۳ _NORM_ARRIVAL_RATE بازکالیبره", abs(state_builder._NORM_ARRIVAL_RATE - 20.0) > 1)

import algorithms.ppo.env as env_mod
check("۱۱.۴ _NORM_REJECTED_PER_TICK بازکالیبره یا تأییدشده", env_mod._NORM_REJECTED_PER_TICK == 6.0)

# --- خلاصه ---
print("\n" + "="*50)
total = len(checks)
passed = sum(1 for _, ok in checks if ok)
print(f"نتیجه: {passed}/{total} فیکس تأیید شد")
if passed < total:
    print("⚠️ قبل از آموزش، موارد ✗ بالا را برطرف کن!")
else:
    print("✅ همه‌ی فیکس‌ها موجودند — آماده‌ی آموزش نهایی")

[✗ MISSING] ۱.۱ DRAINING در utilization
[✓] ۱.۲ load_balance_cv با ۱ سرور فعال
[✓] ۲.۱ capacity_starved شامل BOOTING
[✓] ۲.۱ threshold پارامتری شده
[✓] ۲.۲ Voila با OCC_UP_THRESHOLD خودش
[✓] ۲.۳ engine._any_service_capacity_starved شامل BOOTING
[✓] ۳ n_ready_replicas در snapshot معمولی
[✓] ۳ n_ready_replicas در snapshot readonly
[✓] ۴.۱ mask can_up چک ACTIVE
[✓] ۴.۱ mask can_down از n_ready/mature_ready
[✓] ۴.۲ ppo_algorithm mask چک ACTIVE
[✓] ۵ demand_centroid فقط یک‌بار append میشه (نه دوبار)
[✓] ۶.۱ ENERGY_RESYNC در EventType
[✓] ۶.۲ ENERGY_RESYNC در step()
[✓] ۶.۳ drain دینامیک (available_at)
[✓] ۷.۱ _was_turn_on_necessary_audit موجود
[✓] ۷.۲ _was_turn_off_necessary_audit موجود
[SKIP] k8s_adapter نیاز به کتابخانه‌ی kubernetes/redis دارد - چک نشد
[✓] ۹.۱ رفع bias در env.py (نه break ساده)
[✓] ۹.۲ رفع bias در ppo_algorithm.py
[✓] ۱۰ محافظت سن replica (created_at/mature)
[✓] ۱۱.۱ _NORM_RESPONSE_TIME_SEC بازکالیبره
[✓] ۱۱.۲ _NORM_ENERGY_JOULE بازکالیبره
[✓] ۱۱.۳ _NORM_ARRIVAL_RATE بازک